In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

# Crear el driver de Chrome
driver = webdriver.Chrome()

url = "https://www.ligaprofesional.ar/torneo-apertura-2026/"
driver.get(url)

wait = WebDriverWait(driver, 20)

tables = wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "table")
    )
)


In [5]:

html_content = driver.page_source

In [6]:
from bs4 import BeautifulSoup as BS

soup = BS(html_content)


In [7]:
#driver.close()

In [8]:
tables = soup.find_all("table")
tables[0]

<table><tbody><tr><td class="Opta-title" colspan="7"><h4><span>jueves 22 enero 2026</span></h4></td></tr></tbody><tbody class="Opta-fixture Opta-Match-2612782 Opta-result" data-date="1769112000000" data-match="2612782" data-period="FullTime"><tr class="Opta-Scoreline Opta-Odd"><td class="Opta-Outer Opta-Time"><abbr title="Tiempo completo">TC</abbr></td><td class="Opta-Team Opta-TeamName Opta-Home Opta-Team-8621">Aldosivi</td><td class="Opta-Score Opta-Home Opta-Team-8621 Opta-Team-Left"><span class="Opta-Team-Score">0</span></td><td class="Opta-Divider Opta-Dash">v</td><td class="Opta-Score Opta-Away Opta-Team-8625 Opta-Team-Right"><span class="Opta-Team-Score">0</span></td><td class="Opta-Team Opta-Away Opta-TeamName Opta-Team-8625">Defensa</td><td class="Opta-Outer"></td></tr><tr class="Opta-agg Opta-Odd"><td></td><td colspan="5"><a class="Opta-MatchLink Opta-Ext" href="/ficha-partido?competition=384&amp;season=2026&amp;match=2612782">Página del partido</a></td><td></td></tr></tbody>

In [9]:
import pandas as pd

all_results = []

def output_match(match_dict):
    if match_dict["goles_local"] > match_dict["goles_visitante"]:
        return (match_dict["local"], match_dict["visitante"], 1)
    elif match_dict["goles_local"] < match_dict["goles_visitante"]:
        return (match_dict["local"], match_dict["visitante"], -1)
    else:
        return (match_dict["local"], match_dict["visitante"], 0)

for fecha in range(16):

    #try:

        results = pd.read_html(str(tables[fecha]))[0]
        results = results[results[0] == "TC"].copy()
        results.dropna(axis = 1, how = "all", inplace=True)
        results.rename(columns={1: "local", 2: "goles_local", 4: "goles_visitante", 5: "visitante"}, inplace = True)
        results.reset_index(drop = True, inplace=True)
    
        all_results += [output_match(results.iloc[i].to_dict()) for i in range(results.shape[0])]

    #except:
    #    pass

In [10]:
import json 

with open("results.json", "w") as fp: 
    fp.write(json.dumps(all_results))

In [11]:
from datetime import datetime

str(datetime.today())

'2026-03-25 23:37:25.145399'

In [12]:
equipos = set([item for sublist in all_results for item in sublist[:2]])
N = len(equipos)

name_to_index = {name: i for i, name in enumerate(equipos)}
index_to_name = {value: key for key, value in name_to_index.items()}

matches = [(name_to_index[m[0]], name_to_index[m[1]], m[2]) for m in all_results]
matches

[(7, 6, 0),
 (0, 21, 0),
 (16, 20, 0),
 (29, 26, -1),
 (10, 4, -1),
 (25, 11, -1),
 (17, 23, 0),
 (28, 8, 1),
 (2, 5, 1),
 (9, 12, -1),
 (14, 22, 1),
 (13, 27, -1),
 (24, 18, 1),
 (1, 3, 1),
 (19, 15, 1),
 (20, 10, 1),
 (4, 2, 1),
 (26, 25, -1),
 (21, 28, -1),
 (8, 29, 0),
 (5, 17, 0),
 (7, 9, 0),
 (22, 13, -1),
 (12, 14, 1),
 (23, 24, 1),
 (18, 6, -1),
 (27, 19, 0),
 (11, 16, 1),
 (15, 1, 0),
 (3, 0, 1),
 (25, 29, 1),
 (17, 4, 0),
 (8, 21, 0),
 (2, 20, -1),
 (9, 18, 0),
 (24, 5, 1),
 (13, 12, 0),
 (14, 7, 1),
 (6, 23, 0),
 (19, 22, 1),
 (1, 27, 0),
 (16, 26, 1),
 (0, 15, 1),
 (28, 3, 1),
 (10, 11, 0),
 (29, 16, 1),
 (7, 13, 0),
 (22, 1, 1),
 (12, 19, -1),
 (27, 0, 1),
 (5, 6, -1),
 (20, 17, -1),
 (3, 8, 1),
 (21, 25, 1),
 (26, 10, 1),
 (4, 24, 1),
 (9, 14, 1),
 (23, 18, 1),
 (11, 2, 0),
 (15, 28, -1),
 (19, 7, 1),
 (1, 12, 1),
 (17, 11, 1),
 (6, 4, 0),
 (16, 25, 0),
 (0, 22, -1),
 (21, 3, 1),
 (2, 26, 1),
 (8, 15, 1),
 (28, 27, -1),
 (14, 23, 0),
 (24, 20, 0),
 (13, 9, 1),
 (10, 29, 1

In [36]:
import numpy as np
from scipy.optimize import minimize


def log_likelihood(scores, matches):
    ll = 0.0
    for i, j, y in matches:
        si = scores[i]
        sj = scores[j]

        zi = np.exp(si)
        zj = np.exp(sj)
        zt = np.exp(0.5 * (si + sj))
        Z = zi + zj + zt

        if y == 1:
            ll += np.log(zi / Z)
        elif y == -1:
            ll += np.log(zj / Z)
        else:  # empate
            ll += np.log(zt / Z)
    return ll


In [37]:
def log_posterior(scores, matches, sigma=1.0):
    ll = log_likelihood(scores, matches)
    prior = -0.5 * np.sum(scores**2) / sigma**2
    return ll + prior


In [38]:
def fit_scores(matches, N, sigma=1.0):
    x0 = np.zeros(N)

    def objective(x):
        return -log_posterior(x, matches, sigma)

    res = minimize(
        objective,
        x0,
        method="L-BFGS-B"
    )

    return res.x

In [39]:
scores_hat = fit_scores(matches, N)

df = pd.DataFrame()
df["Equipo"] = list(equipos)
df["Score"] = [scores_hat[name_to_index[equipo]] for equipo in list(equipos)]

In [40]:
df.sort_values("Score", ascending=False).reset_index(drop = True)

,Equipo,Score
0,Vélez,1.103623
1,Independiente Riv.,0.997362
2,Belgrano,0.946773
3,Tigre,0.904501
4,Estudiantes,0.769251
5,Defensa,0.551603
6,Platense,0.498176
7,Independiente,0.481920
8,Lanús,0.324326
9,Talleres,0.285797


Teniendo en cuenta localia y visitantes

In [41]:
import numpy as np
from scipy.optimize import minimize


def log_posterior(theta, matches, N, sigma=1.0):

    # unpack parameters
    s = theta[:N]
    h_home = theta[N:2*N]
    #h_away = theta[2*N:3*N]

    ll = 0.0

    for i, j, y in matches:

        si = s[i] + h_home[i]
        sj = s[j]# + h_away[j]

        a = np.exp(si)
        b = np.exp(sj)
        c = np.exp(0.5 * (si + sj))
        Z = a + b + c

        if y == 1:
            ll += np.log(a / Z)
        elif y == -1:
            ll += np.log(b / Z)
        else:
            ll += np.log(c / Z)

    # Gaussian prior (ridge regularization)
    prior = -0.5 * np.sum(theta**2) / sigma**2

    return ll + prior


In [42]:

def fit_scores(matches, N, sigma=1.0):
    x0 = np.zeros(2*N)

    def objective(x):
        return -log_posterior(x, matches, N, sigma)

    res = minimize(
        objective,
        x0,
        method="L-BFGS-B"
    )

    return res.x


In [43]:
theta_hat = fit_scores(matches, N)


In [44]:
scores_hat = theta_hat[:N]
h_home_hat = theta_hat[N:2*N]
#h_away_hat = theta_hat[2*N:3*N]


In [45]:
df = pd.DataFrame()
df["Equipo"] = list(equipos)
df["Score"] = [scores_hat[name_to_index[equipo]] for equipo in list(equipos)]
df["H_home"] = [h_home_hat[name_to_index[equipo]] for equipo in list(equipos)]
#df["H_away"] = [h_away_hat[name_to_index[equipo]] for equipo in list(equipos)]

In [47]:
df.sort_values("Score", ascending=False).reset_index(drop = True)

,Equipo,Score,H_home
0,Vélez,1.010914,0.750197
1,Belgrano,0.977052,0.207791
2,Independiente Riv.,0.932497,0.302984
3,Tigre,0.859286,0.623465
4,Estudiantes,0.715804,0.712615
5,Platense,0.555609,0.095695
6,Defensa,0.529771,0.123464
7,Independiente,0.394779,0.431721
8,Lanús,0.338406,0.308144
9,Talleres,0.334059,0.146472


In [37]:
def probas(si, sj, h_home, h_away):

    zi = np.exp(si) + h_home
    zj = np.exp(sj) + h_away
    zt = np.exp(0.5 * (si + h_home + sj + h_away))
    Z = zi + zj + zt

    return np.array([zi, zt, zj]) / Z

In [38]:
probas(-0.072, 0.242, 0.427, 0.273)

array([0.30511193, 0.34723826, 0.34764981])